<a href="https://colab.research.google.com/github/rachitjha20/Lead_Scoring_Case_Study/blob/main/Lead_Scoring_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing Libraries

In [85]:
# basic libraries to work on the dataframe
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Importing libraries
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Ignoring Warnings
import warnings
warnings.filterwarnings('ignore')

#Increasing views limit of the columns
pd.options.display.max_columns = None
pd.options.display.max_rows = 150
pd.options.display.float_format = '{:.2f}'.format

## Data understanding and cleaning

### Data Understanding

In [86]:
df = pd.read_csv('Leads.csv')
df.head()

,Prospect ID,Lead Number,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Country,Specialization,How did you hear about X Education,What is your current occupation,What matters most to you in choosing a course,Search,Magazine,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,Receive More Updates About Our Courses,Tags,Lead Quality,Update me on Supply Chain Content,Get updates on DM Content,Lead Profile,City,Asymmetrique Activity Index,Asymmetrique Profile Index,Asymmetrique Activity Score,Asymmetrique Profile Score,I agree to pay the amount through cheque,A free copy of Mastering The Interview,Last Notable Activity
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,API,Olark Chat,No,No,0,0.00,0,0.00,Page Visited on Website,NaN,Select,Select,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Interested in other courses,Low in Relevance,No,No,Select,Select,02.Medium,02.Medium,15.00,15.00,No,No,Modified
1,2a272436-5132-4136-86fa-dcc88c88f482,660728,API,Organic Search,No,No,0,5.00,674,2.50,Email Opened,India,Select,Select,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Ringing,NaN,No,No,Select,Select,02.Medium,02.Medium,15.00,15.00,No,No,Email Opened
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a,660727,Landing Page Submission,Direct Traffic,No,No,1,2.00,1532,2.00,Email Opened,India,Business Administration,Select,Student,Better Career Prospects,No,No,No,No,No,No,No,No,Will revert after reading the email,Might be,No,No,Potential Lead,Mumbai,02.Medium,01.High,14.00,20.00,No,Yes,Email Opened
3,0cc2df48-7cf4-4e39-9de9-19797f9b38cc,660719,Landing Page Submission,Direct Traffic,No,No,0,1.00,305,1.00,Unreachable,India,Media and Advertising,Word Of Mouth,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Ringing,Not Sure,No,No,Select,Mumbai,02.Medium,01.High,13.00,17.00,No,No,Modified
4,3256f628-e534-4826-9d63-4a8b88782852,660681,Landing Page Submission,Google,No,No,1,2.00,1428,1.00,Converted to Lead,India,Select,Other,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Will revert after reading the email,Might be,No,No,Select,Mumbai,02.Medium,01.High,15.00,18.00,No,No,Modified


In [87]:
df.shape

(9240, 37)

In [88]:
df.describe()

,Lead Number,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Asymmetrique Activity Score,Asymmetrique Profile Score
count,9240.00,9240.00,9103.00,9240.00,9103.00,5022.00,5022.00
mean,617188.44,0.39,3.45,487.70,2.36,14.31,16.34
std,23406.00,0.49,4.85,548.02,2.16,1.39,1.81
min,579533.00,0.00,0.00,0.00,0.00,7.00,11.00
25%,596484.50,0.00,1.00,12.00,1.00,14.00,15.00
50%,615479.00,0.00,3.00,248.00,2.00,14.00,16.00
75%,637387.25,1.00,5.00,936.00,3.00,15.00,18.00
max,660737.00,1.00,251.00,2272.00,55.00,18.00,20.00


In [89]:
df.duplicated().sum()

0

In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9240 entries, 0 to 9239
Data columns (total 37 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Prospect ID                                    9240 non-null   object 
 1   Lead Number                                    9240 non-null   int64  
 2   Lead Origin                                    9240 non-null   object 
 3   Lead Source                                    9204 non-null   object 
 4   Do Not Email                                   9240 non-null   object 
 5   Do Not Call                                    9240 non-null   object 
 6   Converted                                      9240 non-null   int64  
 7   TotalVisits                                    9103 non-null   float64
 8   Total Time Spent on Website                    9240 non-null   int64  
 9   Page Views Per Visit                           9103 

#### Observations:

* A large number of null values are present in the columns. These columns should ideally be dropped.
* `Prospect ID` and `Lead Number` are columns that can be ignored, as both serve as unique identifiers. Hence, `Prospect ID` can be removed.
* Some column names contain spaces and are too long. We will be changing them.
* A few columns have `"Select"` as the option, which actually represents a null or NaN value. This occurs when the user doesn't choose anything and leaves the field as it is.

## Data Cleaning

#### Renaming Column Name

In [91]:
# Removing Space and Adding "_"
df.columns =df.columns.str.replace(" ", "_").str.lower()

df.columns

Index(['prospect_id', 'lead_number', 'lead_origin', 'lead_source',
       'do_not_email', 'do_not_call', 'converted', 'totalvisits',
       'total_time_spent_on_website', 'page_views_per_visit', 'last_activity',
       'country', 'specialization', 'how_did_you_hear_about_x_education',
       'what_is_your_current_occupation',
       'what_matters_most_to_you_in_choosing_a_course', 'search', 'magazine',
       'newspaper_article', 'x_education_forums', 'newspaper',
       'digital_advertisement', 'through_recommendations',
       'receive_more_updates_about_our_courses', 'tags', 'lead_quality',
       'update_me_on_supply_chain_content', 'get_updates_on_dm_content',
       'lead_profile', 'city', 'asymmetrique_activity_index',
       'asymmetrique_profile_index', 'asymmetrique_activity_score',
       'asymmetrique_profile_score',
       'i_agree_to_pay_the_amount_through_cheque',
       'a_free_copy_of_mastering_the_interview', 'last_notable_activity'],
      dtype='object')

In [92]:
# Shortern column name length

df.rename(columns = {'totalvisits': 'total_visits', 'total_time_spent_on_website': 'time_on_website',
                    'how_did_you_hear_about_x_education': 'source', 'what_is_your_current_occupation': 'occupation',
                    'what_matters_most_to_you_in_choosing_a_course' : 'course_selection_reason',
                    'receive_more_updates_about_our_courses': 'courses_updates',
                     'update_me_on_supply_chain_content': 'supply_chain_content_updates',
                    'get_updates_on_dm_content': 'dm_content_updates',
                    'i_agree_to_pay_the_amount_through_cheque': 'cheque_payment',
                    'a_free_copy_of_mastering_the_interview': 'mastering_interview'}, inplace = True)

df.head(1)

,prospect_id,lead_number,lead_origin,lead_source,do_not_email,do_not_call,converted,total_visits,time_on_website,page_views_per_visit,last_activity,country,specialization,source,occupation,course_selection_reason,search,magazine,newspaper_article,x_education_forums,newspaper,digital_advertisement,through_recommendations,courses_updates,tags,lead_quality,supply_chain_content_updates,dm_content_updates,lead_profile,city,asymmetrique_activity_index,asymmetrique_profile_index,asymmetrique_activity_score,asymmetrique_profile_score,cheque_payment,mastering_interview,last_notable_activity
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,API,Olark Chat,No,No,0,0.00,0,0.00,Page Visited on Website,NaN,Select,Select,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Interested in other courses,Low in Relevance,No,No,Select,Select,02.Medium,02.Medium,15.00,15.00,No,No,Modified


#### Droping prospect_id Column

In [93]:
df.drop('prospect_id', axis=1, inplace = True)

#### Replacing the "Select" With Null Values


In [94]:
df_obj = df.select_dtypes(include= 'object')

# Finding columns which have "Select"

s = lambda x: x.str.contains('Select', na = False)
# Making list of columns label which have "Select"
l = df_obj.columns[df_obj.apply(s).any()].tolist()
print(l)

['specialization', 'source', 'lead_profile', 'city']


In [95]:
sel_obj = ['specialization', 'source', 'lead_profile', 'city']
df[sel_obj] = df[sel_obj].replace('Select', np.NaN)

#### Handeling the null Values

* Since several columns have a significant number of null entries, let's first calculate the percentage of null values in each column and then make a decision based on the results.
* Additionally, we can drop the sales-generated columns, as they contain data recorded after the sales team has contacted the student. This data is not relevant to the purpose of our model, which is to provide a lead score. The columns to be dropped are:

  - `tags`
  - `lead_quality`
  - `All asymmetrique columns`
  - `last_activity`
  - `last_notable_activity`

In [96]:
df.isnull().mean()*100

,0
lead_number,0.00
lead_origin,0.00
lead_source,0.39
do_not_email,0.00
do_not_call,0.00
converted,0.00
total_visits,1.48
time_on_website,0.00
page_views_per_visit,1.48
last_activity,1.11


##### Observation
- As we can see that there are multiple columns with missing data. Since there are no ways to get the data back from reliable source, so we will be dropping all the columns which have Null value percentage equal or above 40%

#### Droping Columns with Null Value >= 40%

In [97]:
for column in df.columns:
  if df[column].isnull().mean()*100 >= 40:
    df.drop(column, axis = 1, inplace = True)
    print(f"!!! Dropped {column}")

!!! Dropped source
!!! Dropped lead_quality
!!! Dropped lead_profile
!!! Dropped asymmetrique_activity_index
!!! Dropped asymmetrique_profile_index
!!! Dropped asymmetrique_activity_score
!!! Dropped asymmetrique_profile_score


In [98]:
x = ['tags', 'last_activity', 'last_notable_activity']
df.drop(x, axis=1, inplace=True)
print(f"!!! Dropped columns: {x}")

!!! Dropped columns: ['tags', 'last_activity', 'last_notable_activity']
